Reproduces the SAM code (Foret et al. 2021) on Colab.

Before running anything:
Runtime → Change runtime type → T4 GPU.

Run cells in order. Cell 3 restarts the runtime — that is expected;

continue from Cell 4 and do not rerun Cell 3. Setup takes ~10–15 min.

In [ ]:
!nvidia-smi

In [ ]:
%mkdir -p /content/work
%cd /content/work
!git clone -b repro https://github.com/Siddharth-magesh/sam-replication.git sam
!ls sam

THE RUNTIME WILL RESTART

After the restart, continue from Cell 4. Don’t rerun Cell 3.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!mamba create -n sam python=3.8 -y
!mamba install -n sam -c conda-forge cudatoolkit=11.0 "cudnn=8.0.5" -y
!ls /usr/local/envs/sam/lib | grep "libcudnn.so"

The last line must list libcudnn.so.8. If the install errors, run the original two-step version instead:

In [ ]:
!mamba install -n sam -c conda-forge cudatoolkit=11.0 cudnn -y
!mamba install -n sam -c conda-forge "cudnn=8.0.5" -y

The jaxlib line must read 0.1.57+cuda110, not plain 0.1.57.

In [ ]:
PY = "/usr/local/envs/sam/bin/python"

# drop jaxlib, relax the absl-py pin (0.11.0 conflicts with tensorflow-metadata)
!sed -e '/^jaxlib/d' -e 's/^absl-py==0.11.0/absl-py==0.10.0/' /content/work/sam/requirements.txt > /tmp/req_fixed.txt
!grep -E '^(absl|jax)' /tmp/req_fixed.txt

# GPU jaxlib FIRST, from the CUDA-specific index
!{PY} -m pip install "jaxlib==0.1.57+cuda110" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

!{PY} -m pip install -r /tmp/req_fixed.txt
!{PY} -m pip list 2>/dev/null | grep -i -E "^(jax|jaxlib|flax|numpy|tensorflow|absl)"

In [ ]:
import os, glob
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

cands = (glob.glob("/usr/local/envs/sam/**/libdevice*.bc", recursive=True)
         or glob.glob("/usr/local/cuda*/nvvm/libdevice/libdevice*.bc"))
print(cands)
os.makedirs("/usr/local/cuda-11.0/nvvm/libdevice", exist_ok=True)
dst = "/usr/local/cuda-11.0/nvvm/libdevice/libdevice.10.bc"
if not os.path.exists(dst):
    os.symlink(cands[0], dst)
print("libdevice link ok:", os.path.exists(dst))

In [ ]:
!/usr/local/envs/sam/bin/python -c "import jax, jax.numpy as jnp; print(jax.devices()); x=jnp.ones((4096,4096)); print((x@x).block_until_ready().sum())" 2>&1 | grep -v -E "dso_loader|cudart|Skipping"
!/usr/local/envs/sam/bin/python -c "import jax; print(jax.random.normal(jax.random.PRNGKey(0), (3,)))" 2>&1 | tail -3
!/usr/local/envs/sam/bin/python -c "import jax.numpy as jnp; from jax import lax; x=jnp.ones((1,8,8,3)); k=jnp.ones((3,3,3,4)); print(lax.conv_general_dilated(x,k,(1,1),'SAME',dimension_numbers=('NHWC','HWIO','NHWC')).shape)" 2>&1 | tail -3

Expected: [GpuDevice(id=0)] and 68719480000.0; three random numbers; (1, 8, 8, 4). Don’t continue unless all three work.

In [ ]:
import os, shutil
!mkdir -p /root/tensorflow_datasets
!gdown --folder "https://drive.google.com/drive/folders/1k5Dx5VR6YFmxRf6Id4iNq2gdvYIVbgXj" -O /root/tensorflow_datasets

# if the files landed inside a tfds_cache subfolder, move them up
nested = "/root/tensorflow_datasets/tfds_cache"
if os.path.isdir(nested):
    for n in os.listdir(nested):
        shutil.move(os.path.join(nested, n), "/root/tensorflow_datasets/")

!ls /root/tensorflow_datasets/cifar10/3.0.2

The last line should list train/test tfrecord files and dataset_info.json. If the download fails (quota error or folder not found), the training step will download CIFAR-10 itself, which takes ~38 minutes.

In [ ]:
%cd /content/work
!mkdir -p /content/runs
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sam \
  --num_epochs 1 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.05 --run_seed 0 > /content/runs/smoke_sam.log 2>&1
!tail -5 /content/runs/smoke_sam.log
!grep -h "finished in" /content/runs/smoke_sam.log

The cell shows nothing until it finishes. Don’t interrupt it.

In [ ]:
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sgd \
  --num_epochs 1 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.0 --run_seed 0 > /content/runs/smoke_sgd.log 2>&1
!grep -h "finished in" /content/runs/smoke_sgd.log

--sam_rho 0.0 means plain SGD.

In [ ]:
%%writefile /content/read_tb.py
import glob
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from tensorboard.util import tensor_util
for d in sorted(glob.glob("/content/runs/*/lr_*/wd_*/rho_*/seed_*")):
    ea = EventAccumulator(d, size_guidance={"tensors": 0}); ea.Reload()
    print("==", d)
    for t in ["train_error_rate", "test_error_rate", "test_loss"]:
        ev = ea.Tensors(t)
        print("  ", t, [(e.step, round(float(tensor_util.make_ndarray(e.tensor_proto)), 4)) for e in ev])

In [ ]:
!/usr/local/envs/sam/bin/python /content/read_tb.py 2>&1 | grep -v -E "dso_loader|cudart|Skipping"

Save results to your Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sam_runs
!cp -r /content/runs/* /content/drive/MyDrive/sam_runs/

Open the notebook, Runtime → Change runtime type → T4 GPU, then Save.

Run the cells in order. Cell 3 restarts the runtime. That is normal. Continue from Cell 4 and don’t rerun Cell 3.

Stop and message me if Cell 7 doesn’t show [GpuDevice(id=0)], 68719480000.0 and (1, 8, 8, 4).

Cells 9 and 10 look idle for several minutes. Don’t interrupt them.

Success: both runs print Epoch 0 finished in ... s, and Cell 11 shows a test_error_rate well below 0.9 (ours was roughly 0.35–0.45 after 1–2 epochs; this only checks the pipeline). Send me your two epoch times and the GPU type.
If anything fails, send the last 30 lines of the error, not the whole log.